<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

### RapidFire AI RAG/Context Engineering Tutorial Use Case: SciFact Q&A Chatbot

In [1]:
from pathlib import Path

TRITONAI_BASE_URL = "https://tritonai-api.ucsd.edu/v1"
API_KEY_PATH = next(path for path in [Path("../../api-key.txt"), Path("api-key.txt")] if path.exists())
TRITONAI_API_KEY = API_KEY_PATH.read_text().strip()


In [2]:
from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
import re, json
from typing import List as listtype, Dict, Any

INFO 05-08 11:35:53 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-08 11:35:53 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


##### ⚠️ API Cost Considerations
This notebook runs 4 configurations concurrently on a downsampled dataset of 256 examples.
Estimated Costs:
- Current run (downsampled): \$5 
- Full set: \$45

> 💡 **Tip:** Monitor your API usage to avoid unexpected charges.

### Load Dataset and Rename Columns

In [3]:
import pandas as pd
from datasets import Dataset

DATA_ROOT = next(path for path in [Path("../../data"), Path("data")] if path.exists())
SOURCE_DOCS_DIR = DATA_ROOT / "sourcedocs"
VALIDATION_PATH = DATA_ROOT / "validation" / "validation-set-golden-qa-pairs.json"

with open(VALIDATION_PATH, "r", encoding="utf-8") as f:
    validation_examples = json.load(f)

rag_dataset = Dataset.from_dict(
    {
        "query": [example["question"] for example in validation_examples],
        "query_id": [int(example["question_id"]) for example in validation_examples],
        "reference_answer": [example["reference_answer"] for example in validation_examples],
        "expected_source_files": [
            sorted({evidence["file"] for evidence in example.get("source_evidence", [])})
            for example in validation_examples
        ],
    }
)

pd.DataFrame(rag_dataset).head()


,query,query_id,reference_answer,expected_source_files
0,What are the two knob set generators currently...,1,RapidFire AI currently supports two knob set g...,[configs.rst]
1,How does RapidFire AI's adaptive execution eng...,2,Traditional tools force you to run one config ...,[difference.rst]
2,What parameters does the Experiment constructo...,3,The Experiment constructor accepts three param...,[experiment.rst]
3,What is the difference between RFGridSearch an...,4,RFGridSearch requires each knob to have either...,[configs.rst]
4,How do I set up RapidFire AI for RAG evaluatio...,5,For RAG/context engineering with only closed m...,[walkthroughrag.rst]


### Create Experiment

In [4]:
experiment = Experiment(experiment_name="exp1-scifact-full-evaluation", mode="evals")

An experiment with the same name already exists. Created a new experiment 'exp1-scifact-full-evaluation_1' with Experiment ID: 2 at /home/rabhatia/rapidfireai/rapidfire_experiments/exp1-scifact-full-evaluation_1
Created directory: /home/rabhatia/rapidfireai/logs/exp1-scifact-full-evaluation_1


### Define Partial Multi-Config Knobs for LangChain part of RAG Pipeline using RapidFire AI Wrapper APIs

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_openai import OpenAIEmbeddings
from typing import Dict

batch_size = 32


def custom_template(doc: Document) -> str:
    source_file = Path(doc.metadata.get("source", "unknown")).name
    return f"Source file: {source_file}\n{doc.page_content}"


# CPU-based RAG over the uploaded RapidFire AI documentation snapshot
rag_cpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path=str(SOURCE_DOCS_DIR),
        glob="*.rst",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        sample_seed=1337,
    ),
    text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="gpt2", chunk_size=512, chunk_overlap=32
    ),
    embedding_cfg={
        "class": OpenAIEmbeddings,
        "model": "api-tgpt-embeddings",
        "api_key": TRITONAI_API_KEY,
        "base_url": TRITONAI_BASE_URL,
    },
    # FAISS is an in-memory store and only works in create mode.
    vector_store_cfg={
        "type": "faiss",
    }, # if not set, uses FAISS by default
    search_cfg=List([{"type": "similarity", "k": 10}, {"type": "mmr", "k": 10}]), # 2 different search types
    reranker_cfg={
        "class": CrossEncoderReranker,
        "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
        "model_kwargs": {"device": "cpu"},
        "top_n": 5,
    },
    enable_gpu_search=False,
    document_template=custom_template,
)


### Define Data Processing and Postprocessing Functions

In [6]:
INSTRUCTIONS = """
You are a helpful assistant answering questions about the RapidFire AI OSS documentation.
Use only the retrieved documentation evidence provided by the user. If the evidence is insufficient, say what is missing instead of guessing.
Answer concisely but completely. When useful, mention the source file names that support the answer.
"""


In [7]:
def sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)
    retrieved_documents = [
        [Path(doc.metadata.get("source", "unknown")).name for doc in docs]
        for docs in all_context
    ]
    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"\nQuestion:\n{question}\n\nEvidence:\n{context}\n\nAnswer:",
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Function to postprocess outputs produced by generator model"""
    batch["ground_truth_documents"] = batch["expected_source_files"]
    batch["answer"] = batch["generated_text"]
    return batch


### Define Custom Eval Metrics Functions

In [8]:
import math


def compute_ndcg_at_k(retrieved_docs: listtype, expected_docs: set, k=3):
    """Utility function to compute NDCG@k"""
    relevance = [1 if doc in expected_docs else 0 for doc in retrieved_docs[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [1] * ideal_length + [0] * (k - ideal_length)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0


def compute_rr(retrieved_docs: listtype, expected_docs: set):
    """Utility function to compute Reciprocal Rank (RR) for a single query"""
    rr = 0
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            rr = 1 / (i + 1)
            break
    return rr


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Function to compute retrieval metrics against expected source files."""

    true_positives, precisions, recalls, f1_scores, ndcgs, rrs, hit_rates = 0, [], [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        retrieved_docs = list(dict.fromkeys(pred))
        retrieved_set = set(retrieved_docs[:3])

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(compute_ndcg_at_k(retrieved_docs, expected_set, k=3))
        rrs.append(compute_rr(retrieved_docs, expected_set))
        hit_rates.append(1.0 if expected_set.intersection(retrieved_set) else 0.0)

    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@3": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
        "Hit Rate@3": {"value": sum(hit_rates) / total_queries},
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Function to accumulate eval metrics across all batches"""

    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@3", "MRR", "Hit Rate@3"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                )
                / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }


### Define Partial Multi-Config Knobs for TritonAI Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

In [9]:
# 2 TritonAI GPT configs with different generation budgets
triton_gpt_config1 = RFOpenAIAPIModelConfig(
    client_config={"api_key": TRITONAI_API_KEY, "base_url": TRITONAI_BASE_URL, "max_retries": 2},
    model_config={
        "model": "api-gpt-oss-120b",
        "max_completion_tokens": 4096,
    },
    rpm_limit=10_000, # Request per minute (RPM) needs to be set based on your account tier and the specific model used
    tpm_limit=10_000_000, # Token per minute (TPM) needs to be set based on your account tier and the specific model used
    rag=rag_cpu,
    prompt_manager=None,
)

triton_gpt_config2 = RFOpenAIAPIModelConfig(
    client_config={"api_key": TRITONAI_API_KEY, "base_url": TRITONAI_BASE_URL, "max_retries": 2},
    model_config={
        "model": "api-gpt-oss-120b",
        "max_completion_tokens": 1024,
    },
    rpm_limit=10_000, # Request per minute (RPM) needs to be set based on your account tier and the specific model used
    tpm_limit=2_000_000, # Token per minute (TPM) needs to be set based on your account tier and the specific model used
    rag=rag_cpu,
    prompt_manager=None,
)


config_set = {
    "openai_config": List(
        [triton_gpt_config1, triton_gpt_config2]
    ),  # Each represents 2 configs
    "batch_size": batch_size,
    "preprocess_fn": sample_preprocess_fn,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

### Create Config Group

In [10]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

### Run Multi-Config Evals

In [11]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks

# num_actors: In a CPU-only configuration, set this to the number concurrent actors you want to use for inference.
# Available CPU cores are equally divided across all concurrent actors.
results = experiment.run_evals(
    config_group=config_group,
    dataset=rag_dataset,
    num_actors=2, # Assuming 8 CPU cores are available, 4 cores are assigned to each of the 2 actors
    num_shards=4,
    seed=42,
)

=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,29.2s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.base_url,embedding_cfg.class,embedding_cfg.model,vector_store_cfg.type,search_cfg.fetch_k,search_cfg.k,search_cfg.lambda_mult,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,F1 Score,Hit Rate@3,MRR,NDCG@3,Precision,Processing Time,Recall,Samples Per Second,Samples Processed,Throughput,Total
1,api-gpt-oss-120b,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,32,https://tritonai-api.ucsd.edu/v1,OpenAIEmbeddings,api-tgpt-embeddings,faiss,-,10,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,max_completion_tokens=4096,"20.19% [20.19%, 20.19%]","0.4222 [0.4222, 0.4222]","36.67% [36.67%, 36.67%]","0.3651 [0.3651, 0.3651]","14.07% [14.07%, 14.07%]",214.85 seconds,"38.33% [38.33%, 38.33%]",0.21,45,0.3/s,45
2,api-gpt-oss-120b,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,32,https://tritonai-api.ucsd.edu/v1,OpenAIEmbeddings,api-tgpt-embeddings,faiss,20,10,0.5,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,max_completion_tokens=4096,"21.08% [21.08%, 21.08%]","0.4444 [0.4444, 0.4444]","40.26% [40.26%, 40.26%]","0.3812 [0.3812, 0.3812]","14.81% [14.81%, 14.81%]",190.29 seconds,"39.44% [39.44%, 39.44%]",0.24,45,0.3/s,45
3,api-gpt-oss-120b,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,32,https://tritonai-api.ucsd.edu/v1,OpenAIEmbeddings,api-tgpt-embeddings,faiss,-,10,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,max_completion_tokens=1024,"20.19% [20.19%, 20.19%]","0.4222 [0.4222, 0.4222]","36.67% [36.67%, 36.67%]","0.3651 [0.3651, 0.3651]","14.07% [14.07%, 14.07%]",167.17 seconds,"38.33% [38.33%, 38.33%]",0.27,45,0.3/s,45
4,api-gpt-oss-120b,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,32,https://tritonai-api.ucsd.edu/v1,OpenAIEmbeddings,api-tgpt-embeddings,faiss,20,10,0.5,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,max_completion_tokens=1024,"21.08% [21.08%, 21.08%]","0.4444 [0.4444, 0.4444]","40.26% [40.26%, 40.26%]","0.3812 [0.3812, 0.3812]","14.81% [14.81%, 14.81%]",163.97 seconds,"39.44% [39.44%, 39.44%]",0.27,45,0.3/s,45


### View Results

In [12]:
# Convert results dict to DataFrame
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

results_df

,run_id,model_name,text_splitter,chunk_size,chunk_overlap,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second,Total,Precision,Recall,F1 Score,NDCG@3,MRR,Hit Rate@3
0,1,api-gpt-oss-120b,RecursiveCharacterTextSplitter,512,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...",{'type': 'faiss'},"{'type': 'similarity', 'k': 10}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 4096},45,214.85 seconds,0.21,45,0.140741,0.383333,0.201905,0.365068,0.366667,0.422222
1,2,api-gpt-oss-120b,RecursiveCharacterTextSplitter,512,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...",{'type': 'faiss'},"{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 4096},45,190.29 seconds,0.24,45,0.148148,0.394444,0.210794,0.381185,0.402593,0.444444
2,3,api-gpt-oss-120b,RecursiveCharacterTextSplitter,512,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...",{'type': 'faiss'},"{'type': 'similarity', 'k': 10}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,167.17 seconds,0.27,45,0.140741,0.383333,0.201905,0.365068,0.366667,0.422222
3,4,api-gpt-oss-120b,RecursiveCharacterTextSplitter,512,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...",{'type': 'faiss'},"{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,163.97 seconds,0.27,45,0.148148,0.394444,0.210794,0.381185,0.402593,0.444444


### End Experiment

In [13]:
experiment.end()

Experiment exp1-scifact-full-evaluation_1 ended


### View RapidFire AI Log Files

In [14]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

📄 Log File: /home/rabhatia/rapidfireai/logs/exp1-scifact-full-evaluation_1/rapidfire.log

Last 30 lines of rapidfire.log:
2026-05-08 11:40:08 | QueryProcessingActor-1 | INFO | query_actor.py:198 | [exp1-scifact-full-evaluation_1:QueryProcessingActor-1] Recreated embedding function: OpenAIEmbeddings
2026-05-08 11:40:08 | QueryProcessingActor-1 | INFO | query_actor.py:211 | [exp1-scifact-full-evaluation_1:QueryProcessingActor-1] Using CPU-based FAISS for retrieval (avoids GPU memory conflicts)
2026-05-08 11:40:08 | QueryProcessingActor-1 | INFO | query_actor.py:215 | [exp1-scifact-full-evaluation_1:QueryProcessingActor-1] Deserializing FAISS index for this actor...
2026-05-08 11:40:08 | QueryProcessingActor-1 | INFO | query_actor.py:230 | [exp1-scifact-full-evaluation_1:QueryProcessingActor-1] Created independent FAISS vector store for this actor
2026-05-08 11:40:08 | QueryProcessingActor-1 | INFO | query_actor.py:260 | [exp1-scifact-full-evaluation_1:QueryProcessingActor-1] Recreated re